# Phase 1: Dataset Introduction

## NYC Taxi Fare Prediction Using Big Data Analytics

This notebook introduces the NYC Yellow Taxi Trip Records dataset for January 2024.  
The purpose of this notebook is to inspect the dataset before starting preprocessing, analysis, and modeling.

In this phase, we will:

- Load the dataset
- Display the first rows
- Check the dataset shape
- View column names
- Check data types
- Check missing values
- View basic statistics
- Check the pickup and drop-off date range

In [1]:
# Libraries imported via PySpark cell above

## 1. Load the Dataset

In this step, we load the NYC Yellow Taxi January 2024 dataset from the raw Parquet file.

The dataset is loaded from HDFS: `hdfs://namenode:9000/taxi/raw/`. Since Parquet files are not directly readable like CSV or Excel files, we use Pandas to load the file into a DataFrame.

This step only loads the data. No cleaning or preprocessing is performed.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, min as spark_min, max as spark_max

# Start Spark and load January 2024 from HDFS
spark = SparkSession.builder \
    .appName("Phase1 - Dataset Introduction") \
    .getOrCreate()

HDFS_JAN = "hdfs://namenode:9000/taxi/raw/yellow_tripdata_2024-01.parquet"
df = spark.read.parquet(HDFS_JAN)

print("Dataset loaded successfully.")
print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")


Dataset loaded successfully.
Rows: 2,964,624  |  Columns: 19


## 2. Display the First Rows

In this step, we display the first five rows of the dataset.

This gives an initial view of the taxi trip records and helps us understand the available columns and the type of values stored in the dataset.

No changes are made to the data in this step.

In [3]:
# Display first 5 rows
df.show(5, truncate=True)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

## 2.1 Understanding the Displayed Rows

Each row in this dataset represents one completed Yellow Taxi trip in New York City.

From the first five rows, we can see that the dataset contains trip information such as pickup time, drop-off time, passenger count, trip distance, pickup location, drop-off location, payment type, fare amount, tip amount, and total amount.

### Meaning of Important Columns

| Column | Meaning |
|---|---|
| `VendorID` | A code for the technology provider that submitted the trip record. |
| `tpep_pickup_datetime` | The date and time when the taxi meter started. |
| `tpep_dropoff_datetime` | The date and time when the taxi meter stopped. |
| `passenger_count` | The number of passengers in the taxi trip. |
| `trip_distance` | The trip distance measured in miles. |
| `RatecodeID` | A code representing the rate type used for the trip. |
| `store_and_fwd_flag` | Indicates whether the trip record was stored before being sent to the system. |
| `PULocationID` | The pickup taxi zone ID where the meter started. |
| `DOLocationID` | The drop-off taxi zone ID where the meter stopped. |
| `payment_type` | A numeric code showing how the passenger paid, such as credit card or cash. |
| `fare_amount` | The fare calculated by the taxi meter before tips and extra charges. |
| `extra` | Extra charges added to the trip. |
| `mta_tax` | MTA tax charged on the trip. |
| `tip_amount` | Tip paid by the passenger. |
| `tolls_amount` | Toll charges during the trip. |
| `improvement_surcharge` | Improvement surcharge added to the trip. |
| `total_amount` | Final total amount paid by the passenger. |
| `congestion_surcharge` | Congestion surcharge applied in certain NYC zones. |
| `Airport_fee` | Airport-related fee if applicable. |

The pickup and drop-off locations are stored as numeric zone IDs, not written place names. Later, we can use the Taxi Zone Lookup table to translate these IDs into readable boroughs and zone names.

## 3. Check the Dataset Shape

In this step, we check the shape of the dataset.

The shape shows the number of rows and columns in the January 2024 Yellow Taxi trip records file.

It is important to mention that this file is not the full NYC Taxi dataset. It is only one monthly sample/subset from the larger NYC TLC trip record data. We use this sample in Phase 1 to understand the structure of the data before expanding the project to more months or larger-scale processing later.

No preprocessing or cleaning is performed in this step.

In [4]:
print("Number of rows in January 2024:", df.count())
print("Number of columns:", len(df.columns))


Number of rows in January 2024: 2964624
Number of columns: 19


## 4. Display Column Names

In this step, we display all column names in the January 2024 Yellow Taxi sample.

This helps us understand what information is available in the dataset, such as pickup time, drop-off time, trip distance, passenger count, fare amount, payment type, and pickup/drop-off location IDs.

This step is only for dataset understanding. No preprocessing or cleaning is performed.

In [5]:
print("Column names in the January 2024 dataset:")
for c in df.columns:
    print(c)


Column names in the January 2024 dataset:
VendorID
tpep_pickup_datetime
tpep_dropoff_datetime
passenger_count
trip_distance
RatecodeID
store_and_fwd_flag
PULocationID
DOLocationID
payment_type
fare_amount
extra
mta_tax
tip_amount
tolls_amount
improvement_surcharge
total_amount
congestion_surcharge
Airport_fee


## 5. Check Data Types

In this step, we check the data type of each column in the January 2024 Yellow Taxi sample.

This helps us understand how the dataset is stored internally. For example, pickup and drop-off times should be stored as datetime values, while trip distance and fare amount should be stored as numeric values.

Checking data types is important before later preprocessing and analysis, but no changes are made to the data in this phase.

In [6]:
print("Column data types:")
for name, dtype in df.dtypes:
    print(f"  {name}: {dtype}")


Column data types:
  VendorID: int
  tpep_pickup_datetime: timestamp_ntz
  tpep_dropoff_datetime: timestamp_ntz
  passenger_count: bigint
  trip_distance: double
  RatecodeID: bigint
  store_and_fwd_flag: string
  PULocationID: int
  DOLocationID: int
  payment_type: bigint
  fare_amount: double
  extra: double
  mta_tax: double
  tip_amount: double
  tolls_amount: double
  improvement_surcharge: double
  total_amount: double
  congestion_surcharge: double
  Airport_fee: double


## 6. Check Missing Values

In this step, we check how many missing values exist in each column of the January 2024 Yellow Taxi sample.

This helps us understand the initial data quality before any preprocessing is performed.

At this stage, we are only inspecting the missing values. We are not filling, removing, or modifying any data in Phase 1.

In [7]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when

print("Missing/null values per column:")
missing = df.select([
    spark_sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(c)
    if df.schema[c].dataType.typeName() in ('double','float','integer','long')
    else spark_sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
])
missing.show(vertical=True)


Missing/null values per column:
-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 140162 
 trip_distance         | 0      
 RatecodeID            | 140162 
 store_and_fwd_flag    | 140162 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 140162 
 Airport_fee           | 140162 



### Missing Values Observation

The missing values check shows that most columns do not contain missing values.

However, some columns such as `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee` contain missing values.

Each of these columns has `140,162` missing values out of `2,964,624` total rows.

The missing value percentage is approximately:

`140,162 / 2,964,624 ≈ 4.73%`

This means that around 4.73% of the January 2024 sample has missing values in these fields.

This is useful for the project because it shows that the dataset is real and requires preprocessing in later phases. In Phase 1, we only identify the missing values. We do not remove, fill, or modify them at this stage.

## 7. Basic Statistical Summary

In this step, we display basic statistics for the numeric columns in the January 2024 Yellow Taxi sample.

The summary includes values such as count, mean, standard deviation, minimum, maximum, and quartiles.

This helps us understand the general range of important numerical fields such as passenger count, trip distance, fare amount, tip amount, and total amount.

This step may reveal possible outliers or unusual values, but no cleaning or preprocessing is performed in Phase 1.

In [8]:
# Basic statistics for numeric columns
df.describe().show(truncate=False)


+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|VendorID          |passenger_count   |trip_distance     |RatecodeID       |store_and_fwd_flag|PULocationID      |DOLocationID      |payment_type      |fare_amount       |extra             |mta_tax            |tip_amount        |tolls_amount      |improvement_surcharge|total_amount      |congestion_surcharge|Airport_fee        |
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+----

### Statistical Summary Observation

The statistical summary gives an initial overview of the numerical columns in the January 2024 Yellow Taxi sample.

The dataset contains `2,964,624` records in this monthly sample. Some important values from the summary are:

| Column | Mean | Minimum | Maximum | Observation |
|---|---:|---:|---:|---|
| `passenger_count` | `1.34` | `0` | `9` | Most trips have around 1 passenger, but `0` passengers may need checking later. |
| `trip_distance` | `3.65` miles | `0` | `312,722.3` miles | The average trip distance is reasonable, but the maximum value is extremely large and likely an outlier or data error. |
| `fare_amount` | `$18.18` | `-899` | `$5,000` | The average fare is reasonable, but negative fare values may represent refunds, corrections, or invalid records. |
| `tip_amount` | `$3.34` | `-80` | `$428` | Some tip values are negative or very high, which should be investigated later. |
| `total_amount` | `$26.80` | `-900` | `$5,000` | The final amount includes fare, tips, taxes, and surcharges, but negative and extremely high values suggest possible anomalies. |

The pickup datetime also shows a minimum value of `2002-12-31 22:59:39`, even though this file is supposed to represent January 2024. This suggests that some records may contain abnormal date values.

These observations are useful because they show that the dataset is real and contains data quality issues such as missing values, invalid values, negative amounts, abnormal dates, and outliers.

In Phase 1, we only identify and describe these issues. No cleaning, filtering, or preprocessing is performed in this notebook.

## 8. Check Pickup and Drop-off Date Range

In this step, we check the earliest and latest pickup and drop-off datetime values in the January 2024 Yellow Taxi sample.

Although this file represents January 2024 trip records, checking the date range helps us understand whether all records are actually within the expected time period.

This step is useful because the statistical summary showed that the earliest pickup date is outside January 2024, which may indicate abnormal records.

No records are removed or modified in this phase.

In [9]:
from pyspark.sql.functions import min as spark_min, max as spark_max

date_range = df.select(
    spark_min("tpep_pickup_datetime").alias("earliest_pickup"),
    spark_max("tpep_pickup_datetime").alias("latest_pickup"),
    spark_min("tpep_dropoff_datetime").alias("earliest_dropoff"),
    spark_max("tpep_dropoff_datetime").alias("latest_dropoff")
).collect()[0]

print("Pickup datetime range:")
print("  Earliest:", date_range['earliest_pickup'])
print("  Latest:  ", date_range['latest_pickup'])
print("\nDrop-off datetime range:")
print("  Earliest:", date_range['earliest_dropoff'])
print("  Latest:  ", date_range['latest_dropoff'])


Pickup datetime range:
  Earliest: 2002-12-31 22:59:39
  Latest:   2024-02-01 00:01:15

Drop-off datetime range:
  Earliest: 2002-12-31 23:05:41
  Latest:   2024-02-02 13:56:52


### Date Range Observation

The date range check shows that most records are expected to belong to January 2024. However, the earliest pickup and drop-off timestamps are from `2002-12-31`, which is clearly outside the expected period.

This suggests that the dataset contains abnormal timestamp values. These may be caused by taxi meter errors, reporting system issues, or corrupted date records.

The latest pickup timestamp is `2024-02-01 00:01:15`, and the latest drop-off timestamp is `2024-02-02 13:56:52`. Some records slightly after January may occur because trips can start or end near midnight, or because of reporting delays.

This observation is important for later preprocessing. In Phase 3, records outside the selected analysis period can be filtered or handled properly. In Phase 1, we only identify the issue and do not modify the dataset.